# Alternative Master48 Model

This notebook trains a separate XGBoost match model using all usable columns from `Data/master_dataset_48.csv` as team features, encoded into pairwise match deltas. It then runs the corrected 2026 Monte Carlo simulation and writes results to `Data/simulation_results_master48.csv`.


In [ ]:
import os
import sys
from pathlib import Path

if Path.cwd().name == 'Models':
    os.chdir('..')
sys.path.insert(0, str(Path.cwd()))

from Models.master48_alt_model import (
    prepare_experiment,
    tune_model,
    evaluate_model,
    fit_final_model,
    simulate_tournament,
    compare_with_existing_results,
)

import pandas as pd

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')


In [ ]:
N_TRIALS = 60
N_SIMULATIONS = 50_000
SIMULATION_OUTPUT_PATH = 'Data/simulation_results_master48.csv'


In [ ]:
experiment = prepare_experiment()

print('Retained matches:', experiment.retained_match_stats['retained_matches'])
print('All matches:', experiment.retained_match_stats['all_matches'])
print('Retained ratio:', f"{experiment.retained_match_stats['retained_ratio']:.1%}")
print('Retained by tournament:', experiment.retained_match_stats['retained_by_tournament'])
print('Feature count:', len(experiment.feature_cols))
print('Feature columns:')
print(experiment.feature_cols)
print('Fold sizes:')
print([(year, int(train_mask.sum()), int(test_mask.sum())) for train_mask, test_mask, year in experiment.folds])


In [ ]:
best_params, study = tune_model(experiment, n_trials=N_TRIALS)
cv_df = evaluate_model(experiment, best_params)

print('Best params:')
print(best_params)
print('\nCross-validation:')
display(cv_df)
print('\nMean log-loss:', round(float(cv_df['log_loss'].mean()), 4))
print('Mean accuracy:', round(float(cv_df['accuracy'].mean()), 4))


In [ ]:
final_model = fit_final_model(experiment, best_params)
results_df = simulate_tournament(
    experiment,
    final_model,
    n_simulations=N_SIMULATIONS,
    output_path=SIMULATION_OUTPUT_PATH,
)

print('Top simulation results:')
display(results_df.head(20))


In [ ]:
comparison_df = compare_with_existing_results(results_df)

if comparison_df is None:
    print('No existing Data/simulation_results.csv found for comparison.')
else:
    print('Comparison against current model champion percentages:')
    display(comparison_df[['country_code', 'champion_pct', 'current_model_pct', 'delta_vs_current']].head(20))
